# 02.1 - Recursion

## What is recursion?
A function that **calls itself** on a **smaller version** of the same problem.

Every recursive function needs:
1. **Base case** - the smallest input, answered directly (stops the recursion).
2. **Recursive case** - break the problem down and call yourself on the smaller part.
3. **Progress** - each call must move toward the base case.

> Think: *"If someone already solved the smaller problem for me, how do I finish the job?"* - this is the **leap of faith**.

## Why does it matter?
Recursion is the foundation of **trees, graphs (DFS), divide & conquer (merge sort, quicksort), backtracking and dynamic programming**. If recursion clicks, half of DSA becomes easy.

## Where is it used?
| Real system | Use |
|---|---|
| File explorers, `os.walk`, `du`, search in folders | Folders contain folders |
| Parsing JSON / HTML / programming languages | Nested structures |
| Compilers & interpreters | Evaluating expression trees |
| Fractals, computer graphics | Self-similar shapes |
| Sorting libraries | Merge sort / quicksort |
| AI game engines (chess) | Minimax search |

## 1. The call stack - what really happens
Every call gets its own **frame** on the call stack. Frames pile up until the base case, then return one by one.

In [1]:
def factorial(n, depth=0):
    indent = "    " * depth
    print(f"{indent}factorial({n}) called")
    if n <= 1:                                   # base case
        print(f"{indent}-> base case, return 1")
        return 1
    result = n * factorial(n - 1, depth + 1)     # recursive case
    print(f"{indent}-> return {n} * ... = {result}")
    return result

factorial(4)

factorial(4) called
    factorial(3) called
        factorial(2) called
            factorial(1) called
            -> base case, return 1
        -> return 2 * ... = 2
    -> return 3 * ... = 6
-> return 4 * ... = 24


24

## 2. Classic recursive functions

In [2]:
def sum_list(nums):
    if not nums:
        return 0
    return nums[0] + sum_list(nums[1:])

def reverse(s):
    if len(s) <= 1:
        return s
    return reverse(s[1:]) + s[0]

def count_down(n):
    if n == 0:
        return ["liftoff"]
    return [n] + count_down(n - 1)

def flatten(nested):                       # real-world: nested JSON / folders
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result

def tower_of_hanoi(n, source="A", target="C", spare="B", moves=None):
    if moves is None:
        moves = []
    if n == 0:
        return moves
    tower_of_hanoi(n - 1, source, spare, target, moves)   # move n-1 out of the way
    moves.append((source, target))                        # move the biggest
    tower_of_hanoi(n - 1, spare, target, source, moves)   # put n-1 back on top
    return moves

print(sum_list([1, 2, 3, 4]), reverse("python"), count_down(3))
print(flatten([1, [2, [3, [4, 5]], 6], [[7]]]))
print("Hanoi 3 disks:", tower_of_hanoi(3), "->", len(tower_of_hanoi(10)), "moves for 10 disks (2^n - 1)")

10 nohtyp [3, 2, 1, 'liftoff']
[1, 2, 3, 4, 5, 6, 7]
Hanoi 3 disks: [('A', 'C'), ('A', 'B'), ('C', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('A', 'C')] -> 1023 moves for 10 disks (2^n - 1)


## 3. Recursion tree & memoization
Naive Fibonacci recomputes the same values again and again → O(2ⁿ).
**Memoization** = cache results → each value computed once → O(n). This is the bridge to **Dynamic Programming** (02.7).

In [3]:
from functools import cache

calls = 0
def fib(n):
    global calls
    calls += 1
    return n if n <= 1 else fib(n - 1) + fib(n - 2)

fib(25)
print("naive fib(25) calls:", calls)

memo_calls = 0
@cache
def fib_memo(n):
    global memo_calls
    memo_calls += 1
    return n if n <= 1 else fib_memo(n - 1) + fib_memo(n - 2)

fib_memo(25)
print("memoized fib(25) calls:", memo_calls)

naive fib(25) calls: 242785
memoized fib(25) calls: 26


## 4. Recursion vs iteration, and Python's limit
- Any recursion can be rewritten with a loop + explicit stack.
- Python's default recursion limit is 1000 frames (Jupyter raises it to ~3000) → deep recursion raises `RecursionError`.
- On LeetCode, add `sys.setrecursionlimit(10**6)` if the depth can reach ~10⁴–10⁵, or convert to iteration.

In [4]:
import sys
print("default limit:", sys.getrecursionlimit())

def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

try:
    depth(5000)
except RecursionError as e:
    print("RecursionError:", e)

def depth_iterative(n):
    count = 0
    while n:
        n -= 1
        count += 1
    return count
print(depth_iterative(5000))

default limit: 3000
RecursionError: maximum recursion depth exceeded
5000


---
## Solved problems

### 1. Fibonacci Number - LC #509 (Easy)

In [5]:
from functools import cache

@cache
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

assert [fib(n) for n in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
print("Fibonacci ")

Fibonacci 


### 2. Pow(x, n) - LC #50 (Medium) fast exponentiation
Compute xⁿ in O(log n). (Used in cryptography: RSA uses modular fast exponentiation.)

**Idea:** xⁿ = (x^(n/2))² - halve n each call → O(log n) instead of O(n).

In [6]:
def my_pow(x, n):
    if n < 0:
        return 1 / my_pow(x, -n)
    if n == 0:
        return 1
    half = my_pow(x, n // 2)          # solve the half-size problem once
    return half * half if n % 2 == 0 else half * half * x

assert abs(my_pow(2.0, 10) - 1024) < 1e-9
assert abs(my_pow(2.1, 3) - 9.261) < 1e-9
assert abs(my_pow(2.0, -2) - 0.25) < 1e-9
print("Pow ")

Pow 


### 3. Merge Two Sorted Lists (recursive) - LC #21 (Easy)


**Idea:** The smaller head wins; its `next` is the merge of everything that's left.

In [7]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    """[1,2,3] -> 1 -> 2 -> 3  (test helper, not needed on LeetCode)"""
    dummy = cur = ListNode()
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_py(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def merge(l1, l2):
    if not l1 or not l2:
        return l1 or l2
    if l1.val <= l2.val:
        l1.next = merge(l1.next, l2)
        return l1
    l2.next = merge(l1, l2.next)
    return l2

assert list_to_py(merge(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
print("Merge (recursive) ")

Merge (recursive) 


### 4. K-th Symbol in Grammar - LC #779 (Medium)
Row 1 is `0`. Each row replaces 0→01, 1→10. Return the k-th symbol of row n.

**Idea:** Each symbol comes from a parent in the previous row; the left child equals the parent, the right child is flipped.

In [8]:
def kth_grammar(n, k):
    if n == 1:
        return 0
    parent = kth_grammar(n - 1, (k + 1) // 2)
    is_left_child = k % 2 == 1
    return parent if is_left_child else 1 - parent

assert kth_grammar(1, 1) == 0
assert kth_grammar(2, 1) == 0 and kth_grammar(2, 2) == 1
assert [kth_grammar(4, k) for k in range(1, 9)] == [0, 1, 1, 0, 1, 0, 0, 1]
print("K-th Symbol ")

K-th Symbol 


### 5. Generate Parentheses - LC #22 (Medium) All valid combinations of `n` pairs of parentheses.

**Idea:** Add `(` while we still have some; add `)` only if it won't close more than we opened.

In [9]:
def generate_parenthesis(n):
    result = []
    def build(current, open_count, close_count):
        if len(current) == 2 * n:
            result.append(current)
            return
        if open_count < n:
            build(current + "(", open_count + 1, close_count)
        if close_count < open_count:
            build(current + ")", open_count, close_count + 1)
    build("", 0, 0)
    return result

assert generate_parenthesis(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert generate_parenthesis(1) == ["()"]
print("Generate Parentheses ")

Generate Parentheses 


## More practice on LeetCode
Easy: Power of Two (#231), Power of Three (#326), Reverse String (#344), Climbing Stairs (#70)
Medium: Swap Nodes in Pairs (#24), Different Ways to Add Parentheses (#241), Decode String (#394)

## Key takeaways
- Base case + smaller subproblem + trust the recursion.
- Draw the **recursion tree** to find complexity: (number of calls) × (work per call).
- Repeated subproblems → add `@cache` (memoization) → dynamic programming.

**Next: `02.2-Searching-and-Binary-Search.ipynb`**